# Notebook 46 - KLT drift versus Kalman boundary

This notebook answers the question from notebook 45:

> Is the KLT drift normal, and does it depend on what the Kalman filter chooses?

Short version to test:

- **Final output drift/noise** does depend on Kalman choices.
- **Raw UltraTrack/KLT parity** should be judged against MATLAB `fas_x_original/fas_y_original`, which is copied before the MATLAB state estimator modifies `fas_x/fas_y`.
- In the forward MATLAB run, `process_all_UltraTrack` finishes before `do_state_estimation`, so Kalman does not feed back into the same raw KLT sequence.

This notebook quantifies those boundaries using MATLAB raw pre-Kalman geometry, MATLAB final Kalman-smoothed geometry, notebook 44 sequential KLT, and notebook 45 one-step KLT.

In [1]:
from pathlib import Path
import json
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

import numpy as np
import pandas as pd
from scipy.io import loadmat

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.geometry import line_angles_batch, line_lengths_batch, normalize_angle
from ultrasound_tracker.matlab_compat import metric_row

OUT = ROOT / "results" / "notebook46_klt_drift_vs_kalman_boundary"
OUT.mkdir(parents=True, exist_ok=True)

MATLAB_EXPORT = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat")
MATLAB_SOURCE = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UltraTimTrack.m")
NB44_NPZ = ROOT / "results" / "notebook44_ultratrack_klt_oracle_mask_gate" / "opencv_ultratrack_klt_oracle_mask_features_arrays.npz"
NB44_METRICS = ROOT / "results" / "notebook44_ultratrack_klt_oracle_mask_gate" / "klt_oracle_mask_full_metrics.csv"
NB45_NPZ = ROOT / "results" / "notebook45_ultratrack_klt_one_step_affine_diagnostics" / "opencv_ultratrack_klt_one_step_affine_arrays.npz"
NB45_METRICS = ROOT / "results" / "notebook45_ultratrack_klt_one_step_affine_diagnostics" / "klt_one_step_affine_metrics.csv"

for path in [MATLAB_EXPORT, MATLAB_SOURCE, NB44_NPZ, NB44_METRICS, NB45_NPZ, NB45_METRICS]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Output folder:", OUT)

Output folder: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook46_klt_drift_vs_kalman_boundary


## MATLAB source-order evidence

The important control-flow point is whether Kalman feeds back into the raw KLT propagation. The forward MATLAB path calls UltraTrack first and state estimation afterward. During UltraTrack, MATLAB copies the raw propagated `fas_x/fas_y` into `fas_x_original/fas_y_original` before the state estimator is called.

In [2]:
def source_line(line_number: int) -> str:
    lines = MATLAB_SOURCE.read_text(encoding="utf-8", errors="replace").splitlines()
    return lines[line_number - 1].strip()

source_rows = [
    {
        "line": 1650,
        "meaning": "TimTrack runs first for Hough ROItype.",
        "source": source_line(1650),
    },
    {
        "line": 1664,
        "meaning": "Forward UltraTrack/KLT runs before state estimation.",
        "source": source_line(1664),
    },
    {
        "line": 1669,
        "meaning": "State estimation is called after UltraTrack returns.",
        "source": source_line(1669),
    },
    {
        "line": 1846,
        "meaning": "KLT propagates fascicle from previous fas_x/fas_y inside UltraTrack.",
        "source": source_line(1846),
    },
    {
        "line": 1854,
        "meaning": "Raw KLT fascicle x is copied to fas_x_original before state estimation.",
        "source": source_line(1854),
    },
    {
        "line": 1855,
        "meaning": "Raw KLT fascicle y is copied to fas_y_original before state estimation.",
        "source": source_line(1855),
    },
    {
        "line": 2165,
        "meaning": "State estimation function starts after the tracking function.",
        "source": source_line(2165),
    },
    {
        "line": 2191,
        "meaning": "Fascicle Kalman state estimator modifies the tracked geometry later.",
        "source": source_line(2191),
    },
]
source_evidence = pd.DataFrame(source_rows)
source_path = OUT / "matlab_source_order_evidence.csv"
source_evidence.to_csv(source_path, index=False)
print("Saved:", source_path)
display(source_evidence)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook46_klt_drift_vs_kalman_boundary/matlab_source_order_evidence.csv


,line,meaning,source
0,1650,TimTrack runs first for Hough ROItype.,% Run TimTrack
1,1664,Forward UltraTrack/KLT runs before state estim...,"handles = process_all_UltraTrack(hObject, even..."
2,1669,State estimation is called after UltraTrack re...,"handles = do_state_estimation(hObject, eventda..."
3,1846,KLT propagates fascicle from previous fas_x/fa...,fas_prev = [handles.Region(i).Fascicle(j).fas_...
4,1854,Raw KLT fascicle x is copied to fas_x_original...,handles.Region(i).Fascicle(j).fas_x_original{f...
5,1855,Raw KLT fascicle y is copied to fas_y_original...,handles.Region(i).Fascicle(j).fas_y_original{f...
6,2165,State estimation function starts after the tra...,function[handles] = do_state_estimation(hObjec...
7,2191,Fascicle Kalman state estimator modifies the t...,"handles = state_estimator(handles,f,f-1);"


## Load raw KLT, final Kalman, and Python diagnostics

MATLAB endpoint order is `[deep; superficial]`, so fascicle fields are converted to `[x_sup, y_sup, x_deep, y_deep]`.

In [3]:
mat_root = loadmat(MATLAB_EXPORT, simplify_cells=True)["UTT_numeric_export"]
region = mat_root["Region"]
fascicle = region["Fascicle"]
height = int(mat_root["vidHeight"])
mm_per_px = float(mat_root["ID"]) / height
mat_n = int(mat_root["NumFrames"])

nb44 = np.load(NB44_NPZ, allow_pickle=True)
nb45 = np.load(NB45_NPZ, allow_pickle=True)
nb44_metrics = pd.read_csv(NB44_METRICS)
nb45_metrics = pd.read_csv(NB45_METRICS)


def matlab_pair_series(values: object, n: int | None = None) -> np.ndarray:
    out = []
    for value in np.asarray(values, dtype=object).reshape(-1):
        arr = np.asarray(value, dtype=float).reshape(-1)
        out.append(arr[:2] if arr.size >= 2 else [np.nan, np.nan])
    result = np.asarray(out, dtype=float)
    return result if n is None else result[:n]


def matlab_fascicle_segments(x_field: str, y_field: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(fascicle[x_field], n=n)
    y = matlab_pair_series(fascicle[y_field], n=n)
    return np.column_stack([x[:, 1], y[:, 1], x[:, 0], y[:, 0]])


def normalized_angles_deg(segments: np.ndarray) -> np.ndarray:
    return np.asarray([normalize_angle(v, degrees=True) for v in line_angles_batch(segments, degrees=True)], dtype=float)


mat_raw = matlab_fascicle_segments("fas_x_original", "fas_y_original", n=mat_n)
mat_final = matlab_fascicle_segments("fas_x", "fas_y", n=mat_n)
py_seq = np.asarray(nb44["fascicle_segments"], dtype=float)
py_one_step = np.asarray(nb45["fascicle_segments"], dtype=float)

print({
    "frames": mat_n,
    "mm_per_px": mm_per_px,
    "mat_raw_shape": mat_raw.shape,
    "mat_final_shape": mat_final.shape,
    "py_seq_shape": py_seq.shape,
    "py_one_step_shape": py_one_step.shape,
})

{'frames': 2666, 'mm_per_px': 0.09021352313167261, 'mat_raw_shape': (2666, 4), 'mat_final_shape': (2666, 4), 'py_seq_shape': (2666, 4), 'py_one_step_shape': (2666, 4)}


## Boundary metrics

These comparisons separate four different questions:

1. **MATLAB raw vs MATLAB final**: how much the Kalman/smoother changes KLT.
2. **Python sequential vs MATLAB raw**: whether our sequential KLT matches MATLAB raw KLT.
3. **Python sequential vs MATLAB final**: whether our sequential output accidentally looks more like Kalman output.
4. **Python one-step vs MATLAB raw**: whether local LK/affine estimation is close when drift cannot accumulate.

In [4]:
def metrics_for_segments(label: str, reference: np.ndarray, estimate: np.ndarray) -> list[dict]:
    rows = []
    rows.append(metric_row(f"{label}_angle_deg", normalized_angles_deg(reference), normalized_angles_deg(estimate)))
    rows.append(metric_row(f"{label}_length_px", line_lengths_batch(reference), line_lengths_batch(estimate)))
    rows.append(metric_row(f"{label}_length_mm", line_lengths_batch(reference) * mm_per_px, line_lengths_batch(estimate) * mm_per_px))
    rows.append(metric_row(f"{label}_x_sup_px", reference[:, 0], estimate[:, 0]))
    rows.append(metric_row(f"{label}_y_sup_px", reference[:, 1], estimate[:, 1]))
    rows.append(metric_row(f"{label}_x_deep_px", reference[:, 2], estimate[:, 2]))
    rows.append(metric_row(f"{label}_y_deep_px", reference[:, 3], estimate[:, 3]))
    for row in rows:
        row["comparison_group"] = label
    return rows

rows = []
rows.extend(metrics_for_segments("matlab_raw_vs_final", mat_raw, mat_final))
rows.extend(metrics_for_segments("python_seq_vs_matlab_raw", mat_raw, py_seq))
rows.extend(metrics_for_segments("python_seq_vs_matlab_final", mat_final, py_seq))
rows.extend(metrics_for_segments("python_one_step_vs_matlab_raw", mat_raw[1:], py_one_step[1:]))

boundary_metrics = pd.DataFrame(rows)
cols = ["comparison_group", "comparison", "n", "bias", "mae", "rmse", "corr"]
boundary_metrics = boundary_metrics[cols]
boundary_path = OUT / "klt_kalman_boundary_metrics.csv"
boundary_metrics.to_csv(boundary_path, index=False)
print("Saved:", boundary_path)
display(boundary_metrics)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook46_klt_drift_vs_kalman_boundary/klt_kalman_boundary_metrics.csv


,comparison_group,comparison,n,bias,mae,rmse,corr
0,matlab_raw_vs_final,matlab_raw_vs_final_angle_deg,2666,-0.125586,0.863830,1.069571,9.757923e-01
1,matlab_raw_vs_final,matlab_raw_vs_final_length_px,2666,-35.582284,35.585039,38.328078,9.912122e-01
2,matlab_raw_vs_final,matlab_raw_vs_final_length_mm,2666,-3.210003,3.210252,3.457711,9.912122e-01
3,matlab_raw_vs_final,matlab_raw_vs_final_x_sup_px,2666,15.975650,15.975650,17.463506,8.614149e-01
4,matlab_raw_vs_final,matlab_raw_vs_final_y_sup_px,2666,7.945866,7.951556,8.457458,-9.375116e-16
5,matlab_raw_vs_final,matlab_raw_vs_final_x_deep_px,2666,47.359951,47.362209,49.546905,9.919038e-01
6,matlab_raw_vs_final,matlab_raw_vs_final_y_deep_px,2666,-8.253204,8.341886,9.363320,4.874183e-01
7,python_seq_vs_matlab_raw,python_seq_vs_matlab_raw_angle_deg,2666,-5.070816,5.071523,6.118181,7.095121e-01
8,python_seq_vs_matlab_raw,python_seq_vs_matlab_raw_length_px,2666,88.415009,94.138829,121.015689,6.431650e-01
9,python_seq_vs_matlab_raw,python_seq_vs_matlab_raw_length_mm,2666,7.976229,8.492595,10.917252,6.431650e-01


## Does Kalman explain notebook 44 drift?

If notebook 44's sequential KLT mismatch were mostly because we compared against the wrong target, it should be closer to MATLAB final `fas_x/fas_y` than to MATLAB raw `fas_x_original/fas_y_original`. The table below tests that directly.

In [5]:
target_choice_rows = []
for signal, raw_name, final_name in [
    ("angle_deg", "python_seq_vs_matlab_raw_angle_deg", "python_seq_vs_matlab_final_angle_deg"),
    ("length_px", "python_seq_vs_matlab_raw_length_px", "python_seq_vs_matlab_final_length_px"),
    ("length_mm", "python_seq_vs_matlab_raw_length_mm", "python_seq_vs_matlab_final_length_mm"),
    ("x_sup_px", "python_seq_vs_matlab_raw_x_sup_px", "python_seq_vs_matlab_final_x_sup_px"),
    ("x_deep_px", "python_seq_vs_matlab_raw_x_deep_px", "python_seq_vs_matlab_final_x_deep_px"),
]:
    raw_rmse = float(boundary_metrics.loc[boundary_metrics["comparison"] == raw_name, "rmse"].iloc[0])
    final_rmse = float(boundary_metrics.loc[boundary_metrics["comparison"] == final_name, "rmse"].iloc[0])
    target_choice_rows.append({
        "signal": signal,
        "rmse_vs_matlab_raw_pre_kalman": raw_rmse,
        "rmse_vs_matlab_final_post_kalman": final_rmse,
        "closer_target": "raw_pre_kalman" if raw_rmse < final_rmse else "final_post_kalman",
        "final_minus_raw_rmse": final_rmse - raw_rmse,
    })

target_choice = pd.DataFrame(target_choice_rows)
target_choice_path = OUT / "python_sequential_target_choice.csv"
target_choice.to_csv(target_choice_path, index=False)
print("Saved:", target_choice_path)
display(target_choice)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook46_klt_drift_vs_kalman_boundary/python_sequential_target_choice.csv


,signal,rmse_vs_matlab_raw_pre_kalman,rmse_vs_matlab_final_post_kalman,closer_target,final_minus_raw_rmse
0,angle_deg,6.118181,5.932819,final_post_kalman,-0.185363
1,length_px,121.015689,149.056093,raw_pre_kalman,28.040405
2,length_mm,10.917252,13.446875,raw_pre_kalman,2.529624
3,x_sup_px,15.809669,13.473088,final_post_kalman,-2.336581
4,x_deep_px,126.979571,167.082278,raw_pre_kalman,40.102707


## How much does MATLAB Kalman change raw KLT?

This table gives the answer to the user-level intuition: yes, the MATLAB final output depends heavily on the Kalman/smoother. That is downstream of raw KLT parity.

In [6]:
kalman_effect = boundary_metrics[boundary_metrics["comparison_group"] == "matlab_raw_vs_final"].copy()
kalman_effect["interpretation"] = kalman_effect["comparison"].map({
    "matlab_raw_vs_final_angle_deg": "Kalman changes fascicle angle modestly.",
    "matlab_raw_vs_final_length_px": "Kalman shortens/smooths fascicle length by several mm.",
    "matlab_raw_vs_final_length_mm": "Kalman length correction in physical units.",
    "matlab_raw_vs_final_x_sup_px": "Kalman shifts superficial attachment x.",
    "matlab_raw_vs_final_y_sup_px": "Kalman shifts superficial attachment y.",
    "matlab_raw_vs_final_x_deep_px": "Deep attachment changes because the line is reconstructed from state and aponeuroses.",
    "matlab_raw_vs_final_y_deep_px": "Kalman/reconstruction shifts deep attachment y.",
})
kalman_effect_path = OUT / "matlab_kalman_effect_on_raw_klt.csv"
kalman_effect.to_csv(kalman_effect_path, index=False)
print("Saved:", kalman_effect_path)
display(kalman_effect)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook46_klt_drift_vs_kalman_boundary/matlab_kalman_effect_on_raw_klt.csv


,comparison_group,comparison,n,bias,mae,rmse,corr,interpretation
0,matlab_raw_vs_final,matlab_raw_vs_final_angle_deg,2666,-0.125586,0.863830,1.069571,9.757923e-01,Kalman changes fascicle angle modestly.
1,matlab_raw_vs_final,matlab_raw_vs_final_length_px,2666,-35.582284,35.585039,38.328078,9.912122e-01,Kalman shortens/smooths fascicle length by sev...
2,matlab_raw_vs_final,matlab_raw_vs_final_length_mm,2666,-3.210003,3.210252,3.457711,9.912122e-01,Kalman length correction in physical units.
3,matlab_raw_vs_final,matlab_raw_vs_final_x_sup_px,2666,15.975650,15.975650,17.463506,8.614149e-01,Kalman shifts superficial attachment x.
4,matlab_raw_vs_final,matlab_raw_vs_final_y_sup_px,2666,7.945866,7.951556,8.457458,-9.375116e-16,Kalman shifts superficial attachment y.
5,matlab_raw_vs_final,matlab_raw_vs_final_x_deep_px,2666,47.359951,47.362209,49.546905,9.919038e-01,Deep attachment changes because the line is re...
6,matlab_raw_vs_final,matlab_raw_vs_final_y_deep_px,2666,-8.253204,8.341886,9.363320,4.874183e-01,Kalman/reconstruction shifts deep attachment y.


## Drift accumulation from local one-step error

Notebook 45 showed the local OpenCV affine step is close. Notebook 44 showed sequential drift. This table compares those RMSEs directly: if the one-step error is a small fraction of the sequential error, the remaining problem is tracker state accumulation, not the Kalman filter.

In [7]:
drift_rows = []
for signal, seq_name, one_name in [
    ("angle_deg", "python_seq_vs_matlab_raw_angle_deg", "python_one_step_vs_matlab_raw_angle_deg"),
    ("length_px", "python_seq_vs_matlab_raw_length_px", "python_one_step_vs_matlab_raw_length_px"),
    ("length_mm", "python_seq_vs_matlab_raw_length_mm", "python_one_step_vs_matlab_raw_length_mm"),
    ("x_sup_px", "python_seq_vs_matlab_raw_x_sup_px", "python_one_step_vs_matlab_raw_x_sup_px"),
    ("x_deep_px", "python_seq_vs_matlab_raw_x_deep_px", "python_one_step_vs_matlab_raw_x_deep_px"),
]:
    seq_rmse = float(boundary_metrics.loc[boundary_metrics["comparison"] == seq_name, "rmse"].iloc[0])
    one_rmse = float(boundary_metrics.loc[boundary_metrics["comparison"] == one_name, "rmse"].iloc[0])
    drift_rows.append({
        "signal": signal,
        "sequential_rmse_vs_raw": seq_rmse,
        "one_step_rmse_vs_raw": one_rmse,
        "sequential_minus_one_step_rmse": seq_rmse - one_rmse,
        "one_step_fraction_of_sequential_pct": 100.0 * one_rmse / seq_rmse,
    })

drift_decomp = pd.DataFrame(drift_rows)
drift_path = OUT / "drift_accumulation_decomposition.csv"
drift_decomp.to_csv(drift_path, index=False)
print("Saved:", drift_path)
display(drift_decomp)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook46_klt_drift_vs_kalman_boundary/drift_accumulation_decomposition.csv


,signal,sequential_rmse_vs_raw,one_step_rmse_vs_raw,sequential_minus_one_step_rmse,one_step_fraction_of_sequential_pct
0,angle_deg,6.118181,0.301311,5.816870,4.924846
1,length_px,121.015689,5.167994,115.847695,4.270516
2,length_mm,10.917252,0.466223,10.451029,4.270516
3,x_sup_px,15.809669,1.242615,14.567054,7.859841
4,x_deep_px,126.979571,5.725075,121.254496,4.508658


## Per-frame drift onset

This diagnostic marks where notebook 44's sequential KLT first exceeds practical thresholds. It is useful for the next sequential-state notebook, because those are the earliest frames where our state starts to depart from MATLAB raw KLT.

In [8]:
seq_angle_err = normalized_angles_deg(py_seq) - normalized_angles_deg(mat_raw)
seq_len_err_px = line_lengths_batch(py_seq) - line_lengths_batch(mat_raw)
seq_len_err_mm = seq_len_err_px * mm_per_px
seq_endpoint_abs = np.nanmax(np.abs(py_seq - mat_raw), axis=1)

one_angle_err = normalized_angles_deg(py_one_step) - normalized_angles_deg(mat_raw)
one_len_err_px = line_lengths_batch(py_one_step) - line_lengths_batch(mat_raw)
one_len_err_mm = one_len_err_px * mm_per_px
one_endpoint_abs = np.nanmax(np.abs(py_one_step - mat_raw), axis=1)

per_frame = pd.DataFrame({
    "frame": np.arange(mat_n),
    "seq_angle_error_deg": seq_angle_err,
    "seq_length_error_px": seq_len_err_px,
    "seq_length_error_mm": seq_len_err_mm,
    "seq_max_endpoint_abs_error_px": seq_endpoint_abs,
    "one_step_angle_error_deg": one_angle_err,
    "one_step_length_error_px": one_len_err_px,
    "one_step_length_error_mm": one_len_err_mm,
    "one_step_max_endpoint_abs_error_px": one_endpoint_abs,
})
per_frame_path = OUT / "klt_drift_per_frame_errors.csv"
per_frame.to_csv(per_frame_path, index=False)

thresholds = [
    ("seq_abs_angle_gt_1deg", per_frame["seq_angle_error_deg"].abs() > 1.0),
    ("seq_abs_length_gt_2mm", per_frame["seq_length_error_mm"].abs() > 2.0),
    ("seq_endpoint_gt_10px", per_frame["seq_max_endpoint_abs_error_px"] > 10.0),
    ("one_step_abs_angle_gt_1deg", per_frame["one_step_angle_error_deg"].abs() > 1.0),
    ("one_step_abs_length_gt_2mm", per_frame["one_step_length_error_mm"].abs() > 2.0),
    ("one_step_endpoint_gt_10px", per_frame["one_step_max_endpoint_abs_error_px"] > 10.0),
]
threshold_rows = []
for name, mask in thresholds:
    idx = per_frame.loc[mask, "frame"].to_numpy(dtype=int)
    threshold_rows.append({
        "condition": name,
        "first_frame": int(idx[0]) if len(idx) else None,
        "n_frames": int(len(idx)),
        "fraction_frames": float(len(idx) / mat_n),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_path = OUT / "klt_drift_threshold_onset.csv"
threshold_df.to_csv(threshold_path, index=False)
print("Saved:", per_frame_path)
print("Saved:", threshold_path)
display(threshold_df)

display(
    per_frame.assign(abs_seq_length=lambda df: df["seq_length_error_mm"].abs())
    .sort_values("abs_seq_length", ascending=False)
    .head(10)
)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook46_klt_drift_vs_kalman_boundary/klt_drift_per_frame_errors.csv
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook46_klt_drift_vs_kalman_boundary/klt_drift_threshold_onset.csv


,condition,first_frame,n_frames,fraction_frames
0,seq_abs_angle_gt_1deg,43,2346,0.879970
1,seq_abs_length_gt_2mm,42,1914,0.717929
2,seq_endpoint_gt_10px,41,2549,0.956114
3,one_step_abs_angle_gt_1deg,102,54,0.020255
4,one_step_abs_length_gt_2mm,157,17,0.006377
5,one_step_endpoint_gt_10px,102,184,0.069017


,frame,seq_angle_error_deg,seq_length_error_px,seq_length_error_mm,seq_max_endpoint_abs_error_px,one_step_angle_error_deg,one_step_length_error_px,one_step_length_error_mm,one_step_max_endpoint_abs_error_px,abs_seq_length
2503,2503,-14.551271,251.294037,22.670120,271.292295,-0.012261,0.234159,0.021124,0.215460,22.670120
2502,2502,-14.393479,251.062549,22.649237,270.866053,-0.094549,2.470037,0.222831,2.348799,22.649237
2496,2496,-14.109685,249.276412,22.488103,265.447596,-0.270957,4.084336,0.368462,4.160807,22.488103
2500,2500,-14.252513,249.028127,22.465705,268.037127,-0.103131,0.029999,0.002706,0.802901,22.465705
2499,2499,-14.148508,249.001045,22.463262,267.243597,-0.058793,1.449512,0.130766,1.529147,22.463262
2501,2501,-14.273391,248.739445,22.439662,268.517478,-0.014548,-0.210603,-0.018999,0.584871,22.439662
2497,2497,-14.133193,247.572785,22.334413,265.430512,-0.025788,-1.656685,-0.149455,1.263624,22.334413
2498,2498,-14.089116,247.544419,22.331854,265.714629,0.042684,-0.036500,-0.003293,0.525316,22.331854
2573,2573,-13.007351,245.746715,22.169677,254.039614,0.012676,0.103405,0.009328,0.232558,22.169677
2574,2574,-12.855791,245.571690,22.153887,253.183923,0.108018,-0.042968,-0.003876,0.696890,22.153887


## Decision

The correct interpretation is:

- KLT drift is normal in the sense that raw optical-flow tracking can drift and MATLAB later uses Kalman/smoothing to correct final outputs.
- For parity, raw KLT still needs its own gate because MATLAB's Kalman receives the MATLAB raw KLT prediction. If our raw prediction drifts differently, the MATLAB 2-state Kalman port will be fed the wrong prior.
- Notebook 45 shows local KLT is close, so the next KLT notebook should focus on sequential tracker-state behavior: when points are refreshed, which mask is used for `setPoints`, and whether OpenCV is preserving the point set like `vision.PointTracker`.

In [9]:
raw_final_len_rmse_mm = float(boundary_metrics.loc[boundary_metrics["comparison"] == "matlab_raw_vs_final_length_mm", "rmse"].iloc[0])
seq_raw_len_rmse_mm = float(boundary_metrics.loc[boundary_metrics["comparison"] == "python_seq_vs_matlab_raw_length_mm", "rmse"].iloc[0])
one_raw_len_rmse_mm = float(boundary_metrics.loc[boundary_metrics["comparison"] == "python_one_step_vs_matlab_raw_length_mm", "rmse"].iloc[0])
seq_raw_angle_rmse = float(boundary_metrics.loc[boundary_metrics["comparison"] == "python_seq_vs_matlab_raw_angle_deg", "rmse"].iloc[0])
one_raw_angle_rmse = float(boundary_metrics.loc[boundary_metrics["comparison"] == "python_one_step_vs_matlab_raw_angle_deg", "rmse"].iloc[0])

all_seq_closer_to_raw = bool((target_choice["closer_target"] == "raw_pre_kalman").all())
mostly_seq_closer_to_raw = int((target_choice["closer_target"] == "raw_pre_kalman").sum())

summary = {
    "question": "Does KLT drift depend on Kalman choice?",
    "answer": "Final MATLAB output depends on Kalman, but raw KLT fas_x_original/fas_y_original is produced before state estimation in the forward run.",
    "matlab_source_order": "process_all_UltraTrack runs before do_state_estimation; fas_x_original is copied inside UltraTrack before Kalman modifies fas_x.",
    "matlab_kalman_effect_length_rmse_mm": raw_final_len_rmse_mm,
    "python_sequential_vs_raw_length_rmse_mm": seq_raw_len_rmse_mm,
    "python_one_step_vs_raw_length_rmse_mm": one_raw_len_rmse_mm,
    "python_sequential_vs_raw_angle_rmse_deg": seq_raw_angle_rmse,
    "python_one_step_vs_raw_angle_rmse_deg": one_raw_angle_rmse,
    "signals_where_python_seq_is_closer_to_raw_than_final": mostly_seq_closer_to_raw,
    "signals_checked_for_target_choice": int(len(target_choice)),
    "next_action": "Do a sequential point-refresh/state notebook: compare OpenCV persistent tracker behavior against MATLAB setPoints/re-detect logic and test MATLAB-like refresh after every frame.",
}
summary_path = OUT / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:", summary_path)
print(json.dumps(summary, indent=2))

readiness = pd.DataFrame([
    {
        "gate": "TimTrack mask/doHough",
        "status": "passed",
        "ready_for_next": True,
    },
    {
        "gate": "KLT local one-step affine",
        "status": "near-pass; angle/length pass, strict deep x outliers remain",
        "ready_for_next": True,
    },
    {
        "gate": "KLT sequential raw parity",
        "status": "not passed; mismatch is not explained by Kalman target choice",
        "ready_for_next": False,
    },
    {
        "gate": "MATLAB 2-state Kalman",
        "status": "not started; depends on raw KLT prior and TimTrack measurement",
        "ready_for_next": False,
    },
])
readiness_path = OUT / "readiness.csv"
readiness.to_csv(readiness_path, index=False)
print("Saved:", readiness_path)
display(readiness)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook46_klt_drift_vs_kalman_boundary/summary.json
{
  "question": "Does KLT drift depend on Kalman choice?",
  "answer": "Final MATLAB output depends on Kalman, but raw KLT fas_x_original/fas_y_original is produced before state estimation in the forward run.",
  "matlab_source_order": "process_all_UltraTrack runs before do_state_estimation; fas_x_original is copied inside UltraTrack before Kalman modifies fas_x.",
  "matlab_kalman_effect_length_rmse_mm": 3.4577109736379863,
  "python_sequential_vs_raw_length_rmse_mm": 10.91725162369773,
  "python_one_step_vs_raw_length_rmse_mm": 0.46622294045826423,
  "python_sequential_vs_raw_angle_rmse_deg": 6.118181484715813,
  "python_one_step_vs_raw_angle_rmse_deg": 0.3013109932225762,
  "signals_where_python_seq_is_closer_to_raw_than_final": 3,
  "signals_checked_for_target_choice": 5,
  "next_action": "Do a sequential point-refresh/state notebook: compare OpenCV persistent tracker behavio

,gate,status,ready_for_next
0,TimTrack mask/doHough,passed,True
1,KLT local one-step affine,"near-pass; angle/length pass, strict deep x ou...",True
2,KLT sequential raw parity,not passed; mismatch is not explained by Kalma...,False
3,MATLAB 2-state Kalman,not started; depends on raw KLT prior and TimT...,False
